In [1]:
# Author : Sreeram Chennakrishnan

In [1]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sklearn
sklearn.set_config(transform_output="pandas")
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, ElasticNetCV
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    HistGradientBoostingClassifier
)
from sklearn.neural_network import MLPClassifier


In [2]:
df = pd.read_csv('../data/processed/calfire_all_ds_joined.csv')

In [3]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

df.info(verbose=True, show_counts=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3439 entries, 0 to 3438
Data columns (total 106 columns):
 #    Column                            Non-Null Count  Dtype  
---   ------                            --------------  -----  
 0    fire_incident_id                  3439 non-null   object 
 1    fire_incident_county              3429 non-null   object 
 2    fire_incident_acres_burned        3439 non-null   float64
 3    weather_rmax_mean_8w              3415 non-null   float64
 4    weather_rmin_mean_8w              3415 non-null   float64
 5    weather_sph_mean_8w               3415 non-null   float64
 6    weather_srad_mean_8w              3415 non-null   float64
 7    weather_tmmn_mean_8w              3415 non-null   float64
 8    weather_tmmx_mean_8w              3415 non-null   float64
 9    weather_vs_mean_8w                3415 non-null   float64
 10   weather_bi_mean_8w                3415 non-null   float64
 11   weather_fm100_mean_8w             3415 non-null   floa

In [4]:
df['log_acres'] = np.log1p(df['fire_incident_acres_burned'])

In [5]:
labels = [
    'Very Low 1', 'Very Low 2',
    'Low 1', 'Low 2',
    'Medium 1', 'Medium 2',
    'High 1', 'High 2',
    'Very High 1', 'Very High 2'
]

df['fire_acres_bin'] = pd.qcut(
    df['fire_incident_acres_burned'],
    q=10,
    labels=labels
)

df['fire_acres_bin'].value_counts().sort_index()

fire_acres_bin
Very Low 1     373
Very Low 2     345
Low 1          327
Low 2          343
Medium 1       340
Medium 2       335
High 1         346
High 2         342
Very High 1    344
Very High 2    344
Name: count, dtype: int64

In [6]:

def run_classification_models(
    df,
    feature_cols,
    target_col='log_acres',
    test_size=0.2,
    random_state=42
):
    data = df[feature_cols + [target_col]].copy()
    data = data.dropna(subset=[target_col])

    # Create fire size classes from log_acres
    data['fire_size_class'] = pd.qcut(
        data[target_col],
        q=3,
        labels=['Low', 'Medium', 'High']
    )

    X = data[feature_cols]
    y = data['fire_size_class']

    label_encoder = LabelEncoder()
    y_encoded = label_encoder.fit_transform(y)

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y_encoded,
        test_size=test_size,
        random_state=random_state,
        stratify=y_encoded
    )

    models = {
        "Logistic Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=5000,
                random_state=random_state,
                class_weight="balanced"
            ))
        ]),

        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestClassifier(
                n_estimators=500,
                max_depth=5,
                min_samples_leaf=25,
                min_samples_split=50,
                max_features=0.4,
                random_state=random_state,
                n_jobs=-1,
                class_weight="balanced"
            ))
        ]),

        "Extra Trees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesClassifier(
                n_estimators=500,
                max_depth=5,
                min_samples_leaf=20,
                min_samples_split=40,
                max_features=0.4,
                random_state=random_state,
                n_jobs=-1,
                class_weight="balanced"
            ))
        ]),

        "HistGradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingClassifier(
                random_state=random_state,
                max_iter=300,
                max_depth=4,
                min_samples_leaf=35,
                l2_regularization=10.0
            ))
        ]),

        "Neural Network": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPClassifier(
                hidden_layer_sizes=(64, 32),
                activation="relu",
                solver="adam",
                alpha=0.01,
                learning_rate_init=0.001,
                max_iter=1000,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=25,
                random_state=random_state
            ))
        ])
    }

    try:
        from lightgbm import LGBMClassifier

        models["LightGBM"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMClassifier(
                n_estimators=400,
                learning_rate=0.02,
                max_depth=4,
                num_leaves=15,
                min_child_samples=35,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_alpha=10.0,
                reg_lambda=15.0,
                random_state=random_state,
                n_jobs=-1,
                verbose=-1
            ))
        ])
    except ImportError:
        print("LightGBM is not installed. Skipping LightGBM.")

    try:
        from xgboost import XGBClassifier

        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBClassifier(
                n_estimators=400,
                learning_rate=0.03,
                max_depth=3,
                subsample=0.7,
                colsample_bytree=0.7,
                reg_alpha=10.0,
                reg_lambda=15.0,
                eval_metric="mlogloss",
                random_state=random_state,
                n_jobs=-1
            ))
        ])
    except ImportError:
        print("XGBoost is not installed. Skipping XGBoost.")

    results = []
    fitted_models = {}

    for name, model in models.items():
        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        train_acc = accuracy_score(y_train, y_train_pred)
        test_acc = accuracy_score(y_test, y_test_pred)

        results.append({
            "Model": name,

            "Train Accuracy": train_acc,
            "Test Accuracy": test_acc,

            "Train Precision": precision_score(
                y_train,
                y_train_pred,
                average="weighted",
                zero_division=0
            ),
            "Test Precision": precision_score(
                y_test,
                y_test_pred,
                average="weighted",
                zero_division=0
            ),

            "Train Recall": recall_score(
                y_train,
                y_train_pred,
                average="weighted",
                zero_division=0
            ),
            "Test Recall": recall_score(
                y_test,
                y_test_pred,
                average="weighted",
                zero_division=0
            ),

            "Train F1": f1_score(
                y_train,
                y_train_pred,
                average="weighted",
                zero_division=0
            ),
            "Test F1": f1_score(
                y_test,
                y_test_pred,
                average="weighted",
                zero_division=0
            ),

            "Overfit Gap": train_acc - test_acc
        })

        fitted_models[name] = model

    results_df = (
        pd.DataFrame(results)
        .sort_values("Test F1", ascending=False)
        .reset_index(drop=True)
    )

    return results_df, fitted_models, label_encoder

In [15]:

def run_regression_models(df, feature_cols, target_col='log_acres', test_size=0.2, random_state=42):
    data = df[feature_cols + [target_col]].copy()
    data = data.dropna(subset=[target_col])

    X = data[feature_cols]
    y = data[target_col]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state
    )

    models = {
        "Linear Regression": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", LinearRegression())
        ]),

        "Elastic Net": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", ElasticNetCV(cv=5, random_state=random_state, max_iter=10000))
        ]),

        # --- RANDOM FOREST ANTI-OVERFIT ---
        # 1. Lowered max_depth to stop micro-splitting on exact coordinates.
        # 2. Increased min_samples_leaf so a leaf must capture at least 25 fires to be valid.
        # 3. Restricted max_features to 40% per split to force tree diversity.
        "Random Forest": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(
                n_estimators=500,
                max_depth=5,                 # Aggressively limited from 6
                min_samples_leaf=25,         # Increased from 10 to force generalization
                min_samples_split=50,        # Increased from 20
                max_features=0.4,            # Decreased from 0.6 to reduce feature reliance
                random_state=random_state,
                n_jobs=-1
            ))
        ]),

        "Extra Trees": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", ExtraTreesRegressor(
                n_estimators=500,
                max_depth=5,
                min_samples_leaf=20,
                min_samples_split=40,
                max_features=0.4,
                random_state=random_state,
                n_jobs=-1
            ))
        ]),

        # --- HIST GRADIENT BOOSTING ANTI-OVERFIT ---
        # 1. Max iterations capped to avoid over-learning target residuals.
        # 2. Implemented strict max_depth and min_samples_leaf ceilings.
        "HistGradientBoosting": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", HistGradientBoostingRegressor(
                random_state=random_state,
                max_iter=300,                # Reduced from 500 to prevent late-stage memorization
                max_depth=4,                 # Controlled depth
                min_samples_leaf=35,         # Large leaf minimum size requirement
                l2_regularization=10.0       # Added strong L2 penalty on leaf values
            ))
        ]),

           "Neural Network": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", MLPRegressor(
                hidden_layer_sizes=(64, 32),
                activation="relu",
                solver="adam",
                alpha=0.01,
                learning_rate_init=0.001,
                max_iter=1000,
                early_stopping=True,
                validation_fraction=0.15,
                n_iter_no_change=25,
                random_state=random_state
            ))
        ]),
    }

 

    # --- LIGHTGBM ANTI-OVERFIT ---
    # 1. Max depth combined with limited num_leaves handles tree structural growth.
    # 2. min_child_samples=35 mirrors scikit-learn's leaf requirements.
    # 3. reg_alpha and reg_lambda introduce powerful L1/L2 mathematical shrinkage.
    try:
        from lightgbm import LGBMRegressor
        
        models["LightGBM"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", LGBMRegressor(
                n_estimators=400,
                learning_rate=0.02,          # Dropped from 0.03 to slow down learning rate
                max_depth=4,                 # Strict layer count ceiling
                num_leaves=15,               # Explicitly kept low to force broad splits
                min_child_samples=35,        # Leaves must encompass at least 35 fire instances
                subsample=0.7,               # Train each tree on a random 70% fraction of rows
                colsample_bytree=0.7,        # Train each tree on a random 70% fraction of features
                reg_alpha=10.0,              # Heavy L1 penalty to ignore weak/noisy variance features
                reg_lambda=15.0,             # Heavy L2 penalty to smooth final leaf leaf predictions
                random_state=random_state,
                alpha= 0.90,
                n_jobs=-1,
                verbose=-1


            ))
        ])
    except ImportError:
        print("LightGBM is not installed. Skipping LightGBM.")

    # --- XGBOOST ANTI-OVERFIT ---
    # 1. Kept max_depth thin at 3 (highly generalizable, macro relationships only).
    # 2. reg_alpha and reg_lambda penalties added directly into gradient math.
    try:
        from xgboost import XGBRegressor

        models["XGBoost"] = Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", XGBRegressor(
                n_estimators=400,
                learning_rate=0.03,          # Dropped from 0.05
                max_depth=3,                 # Kept tight to prevent deep branch memory lookups
                subsample=0.7,               # row fraction masking
                colsample_bytree=0.7,        # feature fraction masking
                reg_alpha=10.0,              # L1 Regularization penalty
                reg_lambda=15.0,             # L2 Regularization penalty
                objective="reg:squarederror",
                random_state=random_state,
                n_jobs=-1
            ))
        ])
    except ImportError:
        print("XGBoost is not installed. Skipping XGBoost.")



    # 4. Evaluation Loop
    results = []

    for name, model in models.items():
        model.fit(X_train, y_train)

        y_train_pred = model.predict(X_train)
        y_test_pred = model.predict(X_test)

        results.append({
            "Model": name,

            "Train RMSE": np.sqrt(mean_squared_error(y_train, y_train_pred)),
            "Test RMSE": np.sqrt(mean_squared_error(y_test, y_test_pred)),

            "Train MAE": mean_absolute_error(y_train, y_train_pred),
            "Test MAE": mean_absolute_error(y_test, y_test_pred),

            "Train R2": r2_score(y_train, y_train_pred),
            "Test R2": r2_score(y_test, y_test_pred),

            "Overfit Gap": r2_score(y_train, y_train_pred) - r2_score(y_test, y_test_pred)
        })

    results_df = pd.DataFrame(results).sort_values("Test R2", ascending=False)

    return results_df

In [139]:
# For running regression models

# results_df = run_regression_models(
#     df=df,
#     feature_cols=feature_cols,
#     target_col='log_acres'
# )

# results_df

# For running classification models

# results_df, fitted_models, label_encoder = run_classification_models(
#     df=df,
#     feature_cols=feature_cols,
#     target_col='log_acres'
# )

# results_df

In [7]:
col = [ i for i in list(df.columns) if 'vegetation' in i]

In [8]:
col

['vegetation_fuel_type',
 'vegetation_fuel_variation',
 'vegetation_percent',
 'vegetation_ele']

### Feature engineering

In [9]:
scaler = StandardScaler()
df['t_log_spatial_local_urban_population'] = np.log1p(df['spatial_local_urban_population'])
df['std_spatial_distance_to_road_km']  = scaler.fit_transform(df[['spatial_distance_to_road_km']])
df['std_spatial_distance_to_city_km']  = scaler.fit_transform(df[['spatial_distance_to_city_km']])
df['std_spatial_distance_to_coast_km']  = scaler.fit_transform(df[['spatial_distance_to_coast_km']])
df['std_spatial_distance_to_powerline_km']  = scaler.fit_transform(df[['spatial_distance_to_powerline_km']])
df['std_spatial_local_urban_population']  = scaler.fit_transform(df[['spatial_local_urban_population']])

In [10]:
# df.drop(columns=['t_log_spatial_local_urban_population',
#                  'std_spatial_distance_to_road_km',
#                  'std_spatial_distance_to_city_km',
#                  'std_spatial_distance_to_coast_km',
#                  'std_spatial_distance_to_powerline_km',
#                  'std_spatial_local_urban_population'
#                 ]
#                  ,inplace=True)

In [30]:
county_means = df.groupby('fire_incident_county')['log_acres'].mean()
df['fire_county_historical_mean_log_acres'] = df['fire_incident_county'].map(county_means)

In [ ]:
df['_cum_sum_acres'] = df.groupby('fire_incident_county')['log_acres'].cumsum()
df['_past_sum_acres'] = df['_cum_sum_acres'] - df['log_acres']
df['fire_county_historical_mean_log_acres'] = df['_past_sum_acres'] / df['previous_incident_count_by_county']

In [43]:
list(df.columns)

['fire_incident_id',
 'fire_incident_county',
 'fire_incident_acres_burned',
 'weather_rmax_mean_8w',
 'weather_rmin_mean_8w',
 'weather_sph_mean_8w',
 'weather_srad_mean_8w',
 'weather_tmmn_mean_8w',
 'weather_tmmx_mean_8w',
 'weather_vs_mean_8w',
 'weather_bi_mean_8w',
 'weather_fm100_mean_8w',
 'weather_fm1000_mean_8w',
 'weather_erc_mean_8w',
 'weather_etr_mean_8w',
 'weather_pet_mean_8w',
 'weather_vpd_mean_8w',
 'weather_pr_sum_8w',
 'weather_th_sin_8w',
 'weather_th_cos_8w',
 'weather_rmax_mean_6w',
 'weather_rmin_mean_6w',
 'weather_sph_mean_6w',
 'weather_srad_mean_6w',
 'weather_tmmn_mean_6w',
 'weather_tmmx_mean_6w',
 'weather_vs_mean_6w',
 'weather_bi_mean_6w',
 'weather_fm100_mean_6w',
 'weather_fm1000_mean_6w',
 'weather_erc_mean_6w',
 'weather_etr_mean_6w',
 'weather_pet_mean_6w',
 'weather_vpd_mean_6w',
 'weather_pr_sum_6w',
 'weather_th_sin_6w',
 'weather_th_cos_6w',
 'weather_rmax_mean_4w',
 'weather_rmin_mean_4w',
 'weather_sph_mean_4w',
 'weather_srad_mean_4w',
 'we

In [42]:
# df.drop(columns='fire_county_historical_mean_log_acres', inplace=True)

### Running the Models (Classification)

In [12]:
results_df, fitted_models, label_encoder = run_classification_models(
    df=df,
    feature_cols=feature_cols,
    target_col='log_acres'
)

results_df[['Model','Test Accuracy','Test Precision','Test Recall','Overfit Gap']]

,Model,Test Accuracy,Test Precision,Test Recall,Overfit Gap
0,Extra Trees,0.383721,0.383124,0.383721,0.029220
1,XGBoost,0.380814,0.379589,0.380814,0.037579
2,LightGBM,0.377907,0.376217,0.377907,0.040123
3,Random Forest,0.373547,0.373247,0.373547,0.055025
4,Logistic Regression,0.383721,0.400723,0.383721,-0.004950
5,HistGradientBoosting,0.353198,0.353958,0.353198,0.125901
6,Neural Network,0.372093,0.262423,0.372093,0.006678


### Running the Models (Regression)

In [39]:
feature_cols = [

    # Terrain
    'topography_elevation', 'topography_slope', 'topography_aspect',

    # # Fire danger
    # 'erc_2w_std',    
    # 'vpd_2w_std',
    # 'dsci_current_std', 'hot_dry_windy_proxy',
    # 'eddi90d_1wk_avg',
    # 'spei180d_1wk_avg',
    # 'pdsi_1wk_avg',

    # # Interaction
    # 'elevation_x_erc',

    # # Fuel
    # 'fuel_type_historical_mean_log_acres',

    # # Fire history
    # 'previous_incident_count_by_county',

    # # Human Geography
    't_log_spatial_local_urban_population',
    'spatial_distance_to_road_km',
    'spatial_distance_to_city_km',
    'spatial_distance_to_coast_km',
    'spatial_distance_to_powerline_km',
    'fire_county_historical_mean_log_acres',
    # 'fire_county_historical_mean_days_burnt',

]

In [40]:
results_df = run_regression_models(
    df=df,
    feature_cols=feature_cols,
    target_col='log_acres'
)

results_df

,Model,Train RMSE,Test RMSE,Train MAE,Test MAE,Train R2,Test R2,Overfit Gap
7,XGBoost,1.658643,1.857051,1.275792,1.392596,0.338147,0.183800,0.154346
6,LightGBM,1.668958,1.861994,1.284874,1.405780,0.329889,0.179449,0.150439
0,Linear Regression,1.790059,1.863046,1.367236,1.403149,0.229113,0.178522,0.050591
1,Elastic Net,1.794126,1.868572,1.373099,1.410140,0.225606,0.173641,0.051965
2,Random Forest,1.775034,1.885932,1.363937,1.423928,0.242000,0.158216,0.083784
5,Neural Network,1.736053,1.890972,1.324207,1.413888,0.274926,0.153710,0.121216
4,HistGradientBoosting,1.370897,1.891688,1.054321,1.411735,0.547868,0.153069,0.394798
3,Extra Trees,1.882223,1.935920,1.450938,1.472400,0.147689,0.113000,0.034689


In [34]:
feature_cols = [

# Spatial columns 
# 'spatial_distance_to_road_km',
# 'spatial_distance_to_city_km',
# 'spatial_distance_to_coast_km',
# 'spatial_distance_to_powerline_km',
# 'spatial_local_urban_population',
# 't_log_spatial_local_urban_population',
# 'std_spatial_distance_to_road_km',
# 'std_spatial_distance_to_city_km',
# 'std_spatial_distance_to_coast_km',
# 'std_spatial_distance_to_powerline_km',
# 'std_spatial_local_urban_population',

# Drought Columns
# 'drought_dsci_current',
# 'drought_eddi90d_current',
# 'drought_spei180d_current',
# 'drought_pdsi_current',

# Topography
# 'topography_aspect',
# 'topography_elevation',
# 'topography_first',
# 'topography_hillshade',
# 'topography_slope',

# weather
# 'weather_rmax_mean_2w',
# 'weather_rmin_mean_2w',
# 'weather_sph_mean_2w',
# 'weather_srad_mean_2w',
# 'weather_tmmn_mean_2w',
# 'weather_tmmx_mean_2w',
# 'weather_vs_mean_2w',
# 'weather_bi_mean_2w',
# 'weather_fm100_mean_2w',
# 'weather_fm1000_mean_2w',
# 'weather_erc_mean_2w',
# 'weather_etr_mean_2w',
# 'weather_pet_mean_2w',
# 'weather_vpd_mean_2w',
# 'weather_pr_sum_2w',
# 'weather_th_sin_2w',
# 'weather_th_cos_2w',

# vegetation
# 'vegetation_fuel_type',
# 'vegetation_fuel_variation',
# 'vegetation_percent',
# 'vegetation_ele',

]

In [32]:
feature_cols = [

# Spatial columns 
# 'spatial_distance_to_road_km',
# 'spatial_distance_to_city_km',
# 'spatial_distance_to_coast_km',
# 'spatial_distance_to_powerline_km',
# 'spatial_local_urban_population',
# 't_log_spatial_local_urban_population',
# 'std_spatial_distance_to_road_km',
# 'std_spatial_distance_to_city_km',
# 'std_spatial_distance_to_coast_km',
# 'std_spatial_distance_to_powerline_km',
# 'std_spatial_local_urban_population',

# Drought Columns
# 'drought_dsci_current',
# 'drought_eddi90d_current',
# 'drought_spei180d_current',
# 'drought_pdsi_current',

# Topography
# 'topography_aspect',
'topography_elevation',
# 'topography_first',
# 'topography_hillshade',
# 'topography_slope',

# weather
# 'weather_rmax_mean_2w',
# 'weather_rmin_mean_2w',
# 'weather_sph_mean_2w',
# 'weather_srad_mean_2w',
# 'weather_tmmn_mean_2w',
# 'weather_tmmx_mean_2w',
# 'weather_vs_mean_2w',
# 'weather_bi_mean_2w',
# 'weather_fm100_mean_2w',
# 'weather_fm1000_mean_2w',
# 'weather_erc_mean_2w',
# 'weather_etr_mean_2w',
# 'weather_pet_mean_2w',
# 'weather_vpd_mean_2w',
# 'weather_pr_sum_2w',
# 'weather_th_sin_2w',
# 'weather_th_cos_2w',

# vegetation
# 'vegetation_fuel_type',
# 'vegetation_fuel_variation',
# 'vegetation_percent',
# 'vegetation_ele',

]

In [17]:
df[['log_acres','std_spatial_local_urban_population','t_log_spatial_local_urban_population','spatial_local_urban_population',
    'spatial_distance_to_road_km','std_spatial_distance_to_road_km',
    'spatial_distance_to_city_km','std_spatial_distance_to_city_km',
    'spatial_distance_to_coast_km','std_spatial_distance_to_coast_km',
    'spatial_distance_to_powerline_km','std_spatial_distance_to_powerline_km',
    'spatial_local_urban_population','std_spatial_local_urban_population',

'drought_dsci_current',
'drought_eddi90d_current',
'drought_spei180d_current',
'drought_pdsi_current',
'topography_aspect',
'topography_elevation',
'topography_first',
'topography_hillshade',
'topography_slope',
'weather_rmax_mean_2w',
'weather_rmin_mean_2w',
'weather_sph_mean_2w',
'weather_srad_mean_2w',
'weather_tmmn_mean_2w',
'weather_tmmx_mean_2w',
'weather_vs_mean_2w',
'weather_bi_mean_2w',
'weather_fm100_mean_2w',
'weather_fm1000_mean_2w',
'weather_erc_mean_2w',
'weather_etr_mean_2w',
'weather_pet_mean_2w',
'weather_vpd_mean_2w',
'weather_pr_sum_2w',
'weather_th_sin_2w',
'weather_th_cos_2w',
'vegetation_percent',
'vegetation_ele'
   
   ]].corr()

,log_acres,std_spatial_local_urban_population,t_log_spatial_local_urban_population,spatial_local_urban_population,spatial_distance_to_road_km,std_spatial_distance_to_road_km,spatial_distance_to_city_km,std_spatial_distance_to_city_km,spatial_distance_to_coast_km,std_spatial_distance_to_coast_km,spatial_distance_to_powerline_km,std_spatial_distance_to_powerline_km,spatial_local_urban_population,std_spatial_local_urban_population,drought_dsci_current,drought_eddi90d_current,drought_spei180d_current,drought_pdsi_current,topography_aspect,topography_elevation,topography_first,topography_hillshade,topography_slope,weather_rmax_mean_2w,weather_rmin_mean_2w,weather_sph_mean_2w,weather_srad_mean_2w,weather_tmmn_mean_2w,weather_tmmx_mean_2w,weather_vs_mean_2w,weather_bi_mean_2w,weather_fm100_mean_2w,weather_fm1000_mean_2w,weather_erc_mean_2w,weather_etr_mean_2w,weather_pet_mean_2w,weather_vpd_mean_2w,weather_pr_sum_2w,weather_th_sin_2w,weather_th_cos_2w,vegetation_percent,vegetation_ele
log_acres,1.000000,-0.055137,-0.116386,-0.055137,0.096281,0.096281,0.131528,0.131528,-0.045647,-0.045647,0.058673,0.058673,-0.055137,-0.055137,0.135299,0.091918,-0.071038,-0.083237,0.021245,0.230645,0.249047,-0.002870,0.209005,-0.178929,-0.160991,-0.092082,0.008430,0.040078,0.046864,-0.056761,0.130743,-0.169925,-0.183903,0.178633,0.053504,0.047940,0.092098,-0.059835,0.052293,-0.018035,0.004074,0.111442
std_spatial_local_urban_population,-0.055137,1.000000,0.728716,1.000000,-0.180338,-0.180338,-0.129848,-0.129848,-0.159024,-0.159024,-0.043487,-0.043487,1.000000,1.000000,0.023383,-0.077591,-0.001623,0.045608,0.005606,-0.104439,-0.039069,-0.014898,-0.010016,0.119510,0.146633,0.102578,-0.122364,0.000230,-0.088822,-0.063452,-0.113334,0.141042,0.105463,-0.121722,-0.155653,-0.150175,-0.126911,-0.015068,0.019745,-0.125303,-0.074593,-0.119584
t_log_spatial_local_urban_population,-0.116386,0.728716,1.000000,0.728716,-0.202279,-0.202279,-0.198640,-0.198640,-0.144842,-0.144842,-0.068503,-0.068503,0.728716,0.728716,0.003156,-0.079772,0.004772,0.033729,0.029386,-0.162989,-0.107667,-0.005944,-0.078413,0.141040,0.133316,0.109279,-0.090350,0.000511,-0.060950,-0.007687,-0.096285,0.143816,0.118232,-0.128092,-0.116282,-0.114644,-0.113234,-0.035165,-0.018554,-0.108814,-0.070638,-0.162498
spatial_local_urban_population,-0.055137,1.000000,0.728716,1.000000,-0.180338,-0.180338,-0.129848,-0.129848,-0.159024,-0.159024,-0.043487,-0.043487,1.000000,1.000000,0.023383,-0.077591,-0.001623,0.045608,0.005606,-0.104439,-0.039069,-0.014898,-0.010016,0.119510,0.146633,0.102578,-0.122364,0.000230,-0.088822,-0.063452,-0.113334,0.141042,0.105463,-0.121722,-0.155653,-0.150175,-0.126911,-0.015068,0.019745,-0.125303,-0.074593,-0.119584
spatial_distance_to_road_km,0.096281,-0.180338,-0.202279,-0.180338,1.000000,1.000000,0.520585,0.520585,0.103862,0.103862,0.381018,0.381018,-0.180338,-0.180338,0.039004,0.105090,0.009330,-0.037100,0.057456,0.381101,0.183773,0.028019,0.061045,-0.120242,-0.080325,-0.214469,-0.009494,-0.204822,-0.122348,0.036811,0.047574,-0.076392,-0.051528,0.068783,-0.038471,-0.040265,-0.039889,0.074210,0.129932,0.248951,0.131583,0.279024
std_spatial_distance_to_road_km,0.096281,-0.180338,-0.202279,-0.180338,1.000000,1.000000,0.520585,0.520585,0.103862,0.103862,0.381018,0.381018,-0.180338,-0.180338,0.039004,0.105090,0.009330,-0.037100,0.057456,0.381101,0.183773,0.028019,0.061045,-0.120242,-0.080325,-0.214469,-0.009494,-0.204822,-0.122348,0.036811,0.047574,-0.076392,-0.051528,0.068783,-0.038471,-0.040265,-0.039889,0.074210,0.129932,0.248951,0.131583,0.279024
spatial_distance_to_city_km,0.131528,-0.129848,-0.198640,-0.129848,0.520585,0.520585,1.000000,1.000000,0.107788,0.107788,0.783073,0.783073,-0.129848,-0.129848,0.017561,0.069542,0.005193,-0.021318,-0.013996,0.514028,0.232994,0.001788,0.116220,-0.133942,-0.107898,-0.181896,0.026843,-0.184045,-0.088411,-0.075682,-0.015521,-0.108233,-0.073643,0.076194,-0.031562,-0.020372,-0.012199,0.096294,0.132984,0.115491,0.144285,0.364162
std_spat